<a href="https://colab.research.google.com/github/youngbean0519/Self-Project/blob/main/notebooks/06_textcnn_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/youngbean0519/Self-Project.git
%cd Self-Project/

Cloning into 'Self-Project'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 80 (delta 26), reused 59 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 508.31 KiB | 831.00 KiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/Self-Project


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys

PROJECT_DIR = "/content/Self-Project"
SRC_DIR = os.path.join(PROJECT_DIR, "src")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
DATA_DIR = "/content/drive/MyDrive/26_KOTE/data"
ARTIFACT_DIR = "/content/drive/MyDrive/26_KOTE/artifacts/week5_svm"

sys.path.append(PROJECT_DIR)
sys.path.append(SRC_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("SRC_DIR:", SRC_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("DATA_DIR:", DATA_DIR)
print("ARTIFACT_DIR:", ARTIFACT_DIR)

PROJECT_DIR: /content/Self-Project
SRC_DIR: /content/Self-Project/src
RESULTS_DIR: /content/Self-Project/results
DATA_DIR: /content/drive/MyDrive/26_KOTE/data
ARTIFACT_DIR: /content/drive/MyDrive/26_KOTE/artifacts/week5_svm


In [ ]:
import preprocessing.text as text_preprocessing
import preprocessing.labels as preprocessing_labels
import evaluation.metrics as metrics
import utils.labels as utils_labels

import time
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    f1_score,
    hamming_loss,
    average_precision_score
)

train_path = os.path.join(DATA_DIR, "train.tsv")
val_path = os.path.join(DATA_DIR, "val.tsv")
test_path = os.path.join(DATA_DIR, "test.tsv")

train_df = pd.read_csv(train_path, sep="\t", header=None, names=["id", "text", "labels"])
val_df = pd.read_csv(val_path, sep="\t", header=None, names=["id", "text", "labels"])
test_df = pd.read_csv(test_path, sep="\t", header=None, names=["id", "text", "labels"])

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

NUM_LABELS = 44

train_label_lists = train_df["labels"].apply(preprocessing_labels.parse_label_ids).tolist()
val_label_lists = val_df["labels"].apply(preprocessing_labels.parse_label_ids).tolist()
test_label_lists = test_df["labels"].apply(preprocessing_labels.parse_label_ids).tolist()

y_train, mlb = preprocessing_labels.encode_multilabels(train_label_lists, NUM_LABELS)
y_val = mlb.transform(val_label_lists)
y_test = mlb.transform(test_label_lists)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

def make_preprocessed_texts(df, normalize_repeat, remove_symbols):
  return df["text"].astype(str).apply(
      lambda text: text_preprocessing.preprocess_text(
          text,
          normalize_repeat=normalize_repeat,
          remove_symbols=remove_symbols
      )
  )

preprocessing_conditions = {
    "raw": {
        "normalize_repeat": False,
        "remove_symbols": False
    },

    "repeat": {
        "normalize_repeat": True,
        "remove_symbols": False
    },

    "repeat_symbols": {
        "normalize_repeat": True,
        "remove_symbols": True
    }
}

sample = train_df["text"].astype(str)

sample_repeat = sample.apply(
    lambda x: text_preprocessing.preprocess_text(
        x,
        normalize_repeat=True,
        remove_symbols=False
    )
)

sample_repeat_symbols = sample.apply(
    lambda x: text_preprocessing.preprocess_text(
        x,
        normalize_repeat=True,
        remove_symbols=True
    )
)

print(
    "raw -> repeat changed:",
    (sample != sample_repeat).sum()
)

print(
    "repeat -> repeat_symbols changed:",
    (sample_repeat != sample_repeat_symbols).sum()
)

print(
    "raw -> repeat_symbols changed:",
    (sample != sample_repeat_symbols).sum()
)

train: (40000, 3)
val: (5000, 3)
test: (5000, 3)
y_train: (40000, 44)
y_val: (5000, 44)
y_test: (5000, 44)
raw -> repeat changed: 13144
repeat -> repeat_symbols changed: 25198
raw -> repeat_symbols changed: 28865


In [ ]:
changed_repeat = sample != sample_repeat

for original, processed in zip(
    sample[changed_repeat].head(20),
    sample_repeat[changed_repeat].head(20)
):
    print("ORIGINAL :", original)
    print("REPEAT   :", processed)
    print("-" * 80)

ORIGINAL : 내가 톰행크스를 좋아하긴 했나보다... 초기 영화 빼고는 다 봤네.
REPEAT   : 내가 톰행크스를 좋아하긴 했나보다.. 초기 영화 빼고는 다 봤네.
--------------------------------------------------------------------------------
ORIGINAL : 미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ
REPEAT   : 미역은 원생생물계 산호초는 동물ㅇㅇ 아 미역이 바다의 새ㄱㅇㄱㅋㅋ
--------------------------------------------------------------------------------
ORIGINAL : 어릴 때 했던 건데 아직도 볼 때마다 뒤통수 얼얼함 ㅋㅋㅋㅋ
REPEAT   : 어릴 때 했던 건데 아직도 볼 때마다 뒤통수 얼얼함 ㅋㅋ
--------------------------------------------------------------------------------
ORIGINAL : 물갈이약 구매했어요...미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 그 동안 만족했었는데... 약간 불량들이 있네요... 1) 함께 구매한 스포이드에 구멍이 있네요... 2) 부화통에 같이오는 흡착고무가 불량이라 안붙어요... 3) 예전에 구매한 온도계의 온도가 안맞아요...
REPEAT   : 물갈이약 구매했어요..미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 그 동안 만족했었는데.. 약간 불량들이 있네요.. 1) 함께 구매한 스포이드에 구멍이 있네요.. 2) 부화통에 같이오는 흡착고무가 불량이라 안붙어요.. 3) 예전에 구매한 온도계의 온도가 안맞아요..
--------------------------------------------------------------------------------
ORIGINAL : 스토리 담당만 붙여줬으면 갓작품 됬을텐데.....
RE

In [ ]:
changed_symbols = (
    sample_repeat != sample_repeat_symbols
)

for before, after in zip(
    sample_repeat[changed_symbols].head(20),
    sample_repeat_symbols[changed_symbols].head(20)
):
    print("BEFORE :", before)
    print("AFTER  :", after)
    print("-" * 80)

BEFORE : 내가 톰행크스를 좋아하긴 했나보다.. 초기 영화 빼고는 다 봤네.
AFTER  : 내가 톰행크스를 좋아하긴 했나보다 초기 영화 빼고는 다 봤네
--------------------------------------------------------------------------------
BEFORE : 정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요 ㅠ 생각보다 스트레스 심한 직업군이라는 생각입니다.
AFTER  : 정말 상상을 초월하는 무개념 진상들 상대하다 우울증 공항장애 걸리는 공무원 많아요 ㅠ 생각보다 스트레스 심한 직업군이라는 생각입니다
--------------------------------------------------------------------------------
BEFORE : 새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수함일지도 모르겠습니다. 그걸 가능케 한 사랑에 경의를 표합니다.
AFTER  : 새로운 세상과 조우한 자의 어린아이 같은 반응 어쩌면 회복된 것은 눈이 아닌 순수함일지도 모르겠습니다 그걸 가능케 한 사랑에 경의를 표합니다
--------------------------------------------------------------------------------
BEFORE : 물갈이약 구매했어요..미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 그 동안 만족했었는데.. 약간 불량들이 있네요.. 1) 함께 구매한 스포이드에 구멍이 있네요.. 2) 부화통에 같이오는 흡착고무가 불량이라 안붙어요.. 3) 예전에 구매한 온도계의 온도가 안맞아요..
AFTER  : 물갈이약 구매했어요 미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 그 동안 만족했었는데 약간 불량들이 있네요 1 함께 구매한 스포이드에 구멍이 있네요 2 부화통에 같이오는 흡착고무가 불량이라 안붙어요 3 예전에 구매한 온도계의 온도가 안맞아요

In [ ]:
print("Total samples:", len(sample))
print("repeat -> repeat_symbols changed:", changed_symbols.sum())
print("Difference ratio:", changed_symbols.mean())
print("All identical:", sample_repeat.equals(sample_repeat_symbols))

Total samples: 40000
repeat -> repeat_symbols changed: 25198
Difference ratio: 0.62995
All identical: False


In [ ]:
comparison_df = pd.DataFrame({
    "original": sample,
    "repeat": sample_repeat,
    "repeat_symbols": sample_repeat_symbols
})

comparison_df[changed_symbols].head(20)

,original,repeat,repeat_symbols
0,내가 톰행크스를 좋아하긴 했나보다... 초기 영화 빼고는 다 봤네.,내가 톰행크스를 좋아하긴 했나보다.. 초기 영화 빼고는 다 봤네.,내가 톰행크스를 좋아하긴 했나보다 초기 영화 빼고는 다 봤네
1,"정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요...","정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요...",정말 상상을 초월하는 무개념 진상들 상대하다 우울증 공항장애 걸리는 공무원 많아요 ...
2,"새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수...","새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수...",새로운 세상과 조우한 자의 어린아이 같은 반응 어쩌면 회복된 것은 눈이 아닌 순수함...
6,물갈이약 구매했어요...미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는...,물갈이약 구매했어요..미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데...,물갈이약 구매했어요 미미네에서 수조2개랑 여러가지 용품사면서 3~4번 주문했었는데 ...
7,"십일조는 겨우60인데? 나머지 다 어디감? 혹시 엄마아빠 그랜져따로타고, 외식자주하...","십일조는 겨우60인데? 나머지 다 어디감? 혹시 엄마아빠 그랜져따로타고, 외식자주하...",십일조는 겨우60인데? 나머지 다 어디감? 혹시 엄마아빠 그랜져따로타고 외식자주하는...
8,90년 줘야 하는거 아닌가? 쓰레기같은 것들.,90년 줘야 하는거 아닌가? 쓰레기같은 것들.,90년 줘야 하는거 아닌가? 쓰레기같은 것들
10,스토리 담당만 붙여줬으면 갓작품 됬을텐데.....,스토리 담당만 붙여줬으면 갓작품 됬을텐데..,스토리 담당만 붙여줬으면 갓작품 됬을텐데
11,"솔직히 말씀드릴게요. 맛없습니다. 장어라서 단백질원이 필요해서 오징어,연어,고등어,...","솔직히 말씀드릴게요. 맛없습니다. 장어라서 단백질원이 필요해서 오징어,연어,고등어,...",솔직히 말씀드릴게요 맛없습니다 장어라서 단백질원이 필요해서 오징어 연어 고등어 쇠고...
13,진작살껄그랬어요~안사고 그냥 넘길려고했는데 아기가 눕기싫어하고 바운서도 이제 싫어해...,진작살껄그랬어요~안사고 그냥 넘길려고했는데 아기가 눕기싫어하고 바운서도 이제 싫어해...,진작살껄그랬어요~안사고 그냥 넘길려고했는데 아기가 눕기싫어하고 바운서도 이제 싫어해...
14,_ 원래 트위터는 배우는 재미가 쏠쏠함ㅋㅋ 일단 님 존재감은 확실히 심어주셨음ㅋㅋㅋ...,_ 원래 트위터는 배우는 재미가 쏠쏠함ㅋㅋ 일단 님 존재감은 확실히 심어주셨음ㅋㅋ ...,_ 원래 트위터는 배우는 재미가 쏠쏠함ㅋㅋ 일단 님 존재감은 확실히 심어주셨음ㅋㅋ ...


In [ ]:
print("Unique texts after repeat:", sample_repeat.nunique())
print("Unique texts after repeat_symbols:", sample_repeat_symbols.nunique())

Unique texts after repeat: 39966
Unique texts after repeat_symbols: 39958


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

TFIDF_PARAMS = {
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_features": 50000,
    "sublinear_tf": True
}

tfidf_repeat = TfidfVectorizer(**TFIDF_PARAMS)
tfidf_repeat_symbols = TfidfVectorizer(**TFIDF_PARAMS)

X_repeat = tfidf_repeat.fit_transform(sample_repeat)
X_repeat_symbols = tfidf_repeat_symbols.fit_transform(sample_repeat_symbols)

print("repeat shape:", X_repeat.shape)
print("repeat_symbols shape:", X_repeat_symbols.shape)

print("repeat vocabulary size:", len(tfidf_repeat.vocabulary_))
print("repeat_symbols vocabulary size:", len(tfidf_repeat_symbols.vocabulary_))

repeat shape: (40000, 50000)
repeat_symbols shape: (40000, 50000)
repeat vocabulary size: 50000
repeat_symbols vocabulary size: 50000


In [ ]:
vocab_repeat = set(tfidf_repeat.vocabulary_.keys())
vocab_repeat_symbols = set(tfidf_repeat_symbols.vocabulary_.keys())

only_repeat = vocab_repeat - vocab_repeat_symbols
only_repeat_symbols = vocab_repeat_symbols - vocab_repeat

print("Only in repeat:", len(only_repeat))
print("Only in repeat_symbols:", len(only_repeat_symbols))

print("\nExamples only in repeat:")
print(list(only_repeat)[:30])

print("\nExamples only in repeat_symbols:")
print(list(only_repeat_symbols)[:30])

Only in repeat: 0
Only in repeat_symbols: 0

Examples only in repeat:
[]

Examples only in repeat_symbols:
[]


In [ ]:
print("Same vocabulary:", vocab_repeat == vocab_repeat_symbols)

Same vocabulary: True


In [ ]:
difference = X_repeat != X_repeat_symbols
print("Number of different TF-IDF values:", difference.nnz)

Number of different TF-IDF values: 0


### Preprocessing Verification Result

The `repeat` and `repeat_symbols` preprocessing methods produced different
text outputs for 25,198 out of 40,000 training samples.

However, after TF-IDF vectorization, both preprocessing conditions produced
the same vocabulary and feature representation.

This suggests that the symbols removed by the `repeat_symbols` condition were
already ignored by the default TF-IDF tokenization process. Therefore, the
identical Logistic Regression and Linear SVM results from Weeks 4 and 5 were
not caused by a preprocessing error.

### 6주차 시작

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score,
    hamming_loss,
    average_precision_score
)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
  torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA L4


In [ ]:
train_texts = make_preprocessed_texts(train_df, normalize_repeat=True, remove_symbols=False)
val_texts = make_preprocessed_texts(val_df, normalize_repeat=True, remove_symbols=False)
test_texts = make_preprocessed_texts(test_df, normalize_repeat=True, remove_symbols=False)

print("Train:", len(train_texts))
print("Validation:", len(val_texts))
print("Test:", len(test_texts))

Train: 40000
Validation: 5000
Test: 5000


In [ ]:
print(train_df.columns)
print(train_df["labels"].iloc[0])

Index(['id', 'text', 'labels'], dtype='object')
2,13,15,16,29,39


In [ ]:
NUM_LABELS = 44

def make_multihot_labels(df, num_labels=44):
    labels = np.zeros(
        (len(df), num_labels),
        dtype=np.float32
    )

    for i, label_str in enumerate(df["labels"].astype(str)):
        label_indices = [
            int(x.strip())
            for x in label_str.split(",")
            if x.strip() != ""
        ]

        labels[i, label_indices] = 1.0

    return labels


y_train = make_multihot_labels(train_df, NUM_LABELS)
y_val = make_multihot_labels(val_df, NUM_LABELS)
y_test = make_multihot_labels(test_df, NUM_LABELS)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

y_train: (40000, 44)
y_val: (5000, 44)
y_test: (5000, 44)


In [ ]:
print("Original labels:")
print(train_df["labels"].iloc[0])

print("\nMulti-hot labels:")
print(y_train[0])

print("\nActive label indices:")
print(np.where(y_train[0] == 1)[0])

Original labels:
2,13,15,16,29,39

Multi-hot labels:
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]

Active label indices:
[ 2 13 15 16 29 39]


In [ ]:
from collections import Counter

def tokenize(text):
  return str(text).split()

In [ ]:
token_counter = Counter()

for text in train_texts:
  token_counter.update(tokenize(text))

print("Number of unique tokens:", len(token_counter))

Number of unique tokens: 216659


In [ ]:
MIN_FREQ = 2

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for token, count in token_counter.items():
  if count >= MIN_FREQ:
    vocab[token] = len(vocab)

PAD_IDX = vocab[PAD_TOKEN]
UNK_IDX = vocab[UNK_TOKEN]

VOCAB_SIZE = len(vocab)

print("Vocabulary size:", VOCAB_SIZE)

Vocabulary size: 41507


In [ ]:
train_lengths = train_texts.apply(lambda x: len(tokenize(x)))

print(train_lengths.describe())

print(
    "95 percentile:",
    np.percentile(train_lengths, 95)
)

print(
    "99 percentile:",
    np.percentile(train_lengths, 99)
)

count    40000.000000
mean        12.420150
std         10.052403
min          1.000000
25%          6.000000
50%          9.000000
75%         16.000000
max         98.000000
Name: text, dtype: float64
95 percentile: 32.0
99 percentile: 52.0


In [ ]:
MAX_LEN = 64

truncated_count = (train_lengths > MAX_LEN).sum()

print("MAX_LEN:", MAX_LEN)
print("Truncated samples:", truncated_count)
print("Truncated ratio:", truncated_count / len(train_lengths))

MAX_LEN: 64
Truncated samples: 112
Truncated ratio: 0.0028


In [ ]:
def encode_text(text, voccab, max_len):
  tokens = tokenize(text)

  ids = [
      vocab.get(token, UNK_IDX)
      for token in tokens
  ]

  ids = ids[:max_len]

  if len(ids) < max_len:
    ids += [PAD_IDX] * (max_len - len(ids))

  return ids

In [ ]:
example_text = train_texts.iloc[0]

print("Text:")
print(example_text)

print("\nEncoded:")
print(
    encode_text(
        example_text,
        vocab,
        MAX_LEN
    )[:30]
)

Text:
내가 톰행크스를 좋아하긴 했나보다.. 초기 영화 빼고는 다 봤네.

Encoded:
[2, 1, 3, 1, 4, 5, 6, 7, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
class KOTEDataset(Dataset):
  def __init__(self, texts, labels, vocab, max_len):
    self.texts = texts.reset_index(drop=True)
    self.labels = labels
    self.vocab = vocab
    self.max_len = max_len

  def __len__(self):
    return len(self.texts)

  def __getitem__(self, idx):
    text = self.texts.iloc[idx]

    input_ids = encode_text(text, self.vocab, self.max_len)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(self.labels[idx], dtype=torch.float32)
    }

In [ ]:
train_dataset = KOTEDataset(train_texts, y_train, vocab, MAX_LEN)
val_dataset = KOTEDataset(val_texts, y_val, vocab, MAX_LEN)
test_dataset = KOTEDataset(test_texts, y_test, vocab, MAX_LEN)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

40000
5000
5000


In [ ]:
BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
batch = next(iter(train_loader))

print("input_ids:", batch["input_ids"].shape)
print("labels:", batch["labels"].shape)

input_ids: torch.Size([128, 64])
labels: torch.Size([128, 44])


In [ ]:
class TextCNN(nn.Module):

  def __init__(
      self,
      vocab_size,
      embed_dim,
      num_filters,
      kernel_sizes,
      num_labels,
      dropout,
      padding_idx
  ):
    super(TextCNN, self).__init__()

    self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)

    self.convs = nn.ModuleList([
        nn.Conv1d(
            in_channels=embed_dim,
            out_channels=num_filters,
            kernel_size=k
        )
        for k in kernel_sizes
    ])

    self.dropout = nn.Dropout(dropout)

    self.fc = nn.Linear(num_filters * len(kernel_sizes), num_labels)

  def forward(self, input_ids):
    x = self.embedding(input_ids)

    # [batch, seq_len, embed_dim]
    # -> [batch, embed_dim, seq_len]
    x = x.permute(0, 2, 1)

    conv_outputs = []

    for conv in self.convs:
      c = torch.relu(conv(x))

      # Global Max Pooling
      c = torch.max(c, dim=2).values

      conv_outputs.append(c)

    x = torch.cat(conv_outputs, dim=1)

    x = self.dropout(x)

    logits = self.fc(x)

    return logits

In [ ]:
EMBED_DIM = 128
NUM_FILTERS = 100
KERNEL_SIZES = [3, 4, 5]
DROPOUT = 0.5

In [ ]:
model = TextCNN(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_filters=NUM_FILTERS,
    kernel_sizes=KERNEL_SIZES,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
    padding_idx=PAD_IDX
).to(device)

print(model)

TextCNN(
  (embedding): Embedding(41507, 128, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(128, 100, kernel_size=(3,), stride=(1,))
    (1): Conv1d(128, 100, kernel_size=(4,), stride=(1,))
    (2): Conv1d(128, 100, kernel_size=(5,), stride=(1,))
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=300, out_features=44, bias=True)
)


In [ ]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
  model.train()

  total_loss = 0.0

  for batch in dataloader:
    input_ids = batch["input_ids"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad()

    logits = model(input_ids)

    loss = criterion(logits, labels)

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

  return total_loss / len(dataloader)

In [ ]:
def predict(model, dataloader, device):
  model.eval()

  all_probs = []
  all_labels = []

  with torch.no_grad():
    for batch in dataloader:
      input_ids = batch["input_ids"].to(device)
      labels = batch["labels"].to(device)

      logits = model(input_ids)
      probs = torch.sigmoid(logits)

      all_probs.append(probs.cpu().numpy())
      all_labels.append(labels.cpu().numpy())

  all_probs = np.vstack(all_probs)
  all_labels = np.vstack(all_labels)

  return all_probs, all_labels

In [ ]:
def evaluate_multilabel(
    y_true,
    y_prob,
    threshold
):
    y_pred = (
        y_prob >= threshold
    ).astype(int)

    micro_f1 = f1_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    hamming = hamming_loss(
        y_true,
        y_pred
    )

    pr_auc_micro = average_precision_score(
        y_true,
        y_prob,
        average="micro"
    )

    pr_auc_macro = average_precision_score(
        y_true,
        y_prob,
        average="macro"
    )

    return {
        "threshold": threshold,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "hamming_loss": hamming,
        "pr_auc_micro": pr_auc_micro,
        "pr_auc_macro": pr_auc_macro
    }

In [ ]:
EPOCHS = 5

history = []

for epoch in range(1, EPOCHS + 1):
  train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)

  val_probs, val_labels = predict(model, val_loader, device)

  result = evaluate_multilabel(val_labels, val_probs, threshold=0.5)

  history.append({
      "epoch": epoch,
      "train_loss": train_loss,
      **result
  })

  print(
      f"Epoch {epoch:02d} | "
      f"Loss: {train_loss:.4f} | "
      f"Micro F1: {result['micro_f1']:.4f} | "
      f"Macro F1: {result['macro_f1']:.4f}"
  )

Epoch 01 | Loss: 0.4439 | Micro F1: 0.1400 | Macro F1: 0.0569
Epoch 02 | Loss: 0.4147 | Micro F1: 0.2328 | Macro F1: 0.1075
Epoch 03 | Loss: 0.3968 | Micro F1: 0.2861 | Macro F1: 0.1393
Epoch 04 | Loss: 0.3835 | Micro F1: 0.3036 | Macro F1: 0.1568
Epoch 05 | Loss: 0.3719 | Micro F1: 0.2992 | Macro F1: 0.1557


In [ ]:
thresholds = np.arange(0.10, 0.61, 0.05)

threshold_results = []

for threshold in thresholds:
    result = evaluate_multilabel(
        val_labels,
        val_probs,
        threshold=threshold
    )
    threshold_results.append(result)

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,threshold,micro_f1,macro_f1,hamming_loss,pr_auc_micro,pr_auc_macro
0,0.10,0.410430,0.331148,0.438973,0.455686,0.314399
1,0.15,0.446462,0.334938,0.331768,0.455686,0.314399
2,0.20,0.465596,0.324031,0.264168,0.455686,0.314399
3,0.25,0.464019,0.296857,0.215655,0.455686,0.314399
4,0.30,0.447652,0.266151,0.190645,0.455686,0.314399
5,0.35,0.418655,0.236682,0.176441,0.455686,0.314399
6,0.40,0.379774,0.207032,0.169327,0.455686,0.314399
7,0.45,0.338326,0.178478,0.166682,0.455686,0.314399
8,0.50,0.299191,0.155724,0.165732,0.455686,0.314399
9,0.55,0.259796,0.133485,0.165977,0.455686,0.314399


In [ ]:
best_idx = threshold_df["micro_f1"].idxmax()
best_result = threshold_df.loc[best_idx]

print("Best threshold:", best_result["threshold"])
print("Micro F1:", best_result["micro_f1"])
print("Macro F1:", best_result["macro_f1"])
print("Hamming Loss:", best_result["hamming_loss"])
print("PR-AUC Micro:", best_result["pr_auc_micro"])
print("PR-AUC Macro:", best_result["pr_auc_macro"])

Best threshold: 0.20000000000000004
Micro F1: 0.4655957186600583
Macro F1: 0.32403119827618343
Hamming Loss: 0.2641681818181818
PR-AUC Micro: 0.4556861876797074
PR-AUC Macro: 0.3143994084822878


## 3. TextCNN with Symbol Removal

The initial TextCNN baseline used the `repeat` preprocessing condition.
Because whitespace tokenization keeps symbols attached to words, symbol
removal may affect the vocabulary and input sequences of TextCNN.

To examine this effect, I trained the same TextCNN model using the
`repeat_symbols` preprocessing condition while keeping all other settings
unchanged.

In [ ]:
train_texts_symbols = make_preprocessed_texts(
    train_df,
    normalize_repeat=True,
    remove_symbols=True
)

val_texts_symbols = make_preprocessed_texts(
    val_df,
    normalize_repeat=True,
    remove_symbols=True
)

test_texts_symbols = make_preprocessed_texts(
    test_df,
    normalize_repeat=True,
    remove_symbols=True
)

In [ ]:
token_counter_symbols = Counter()

for text in train_texts_symbols:
  token_counter_symbols.update(tokenize(text))

vocab_symbols = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for token, count in token_counter_symbols.items():
  if count >= MIN_FREQ:
    vocab_symbols[token] = len(vocab_symbols)

PAD_IDX_SYMBOLS = vocab_symbols[PAD_TOKEN]
UNK_IDX_SYMBOLS = vocab_symbols[UNK_TOKEN]

VOCAB_SIZE_SYMBOLS = len(vocab_symbols)

print("Repeat vocabulary size:", VOCAB_SIZE)
print("Repeat + symbols removed vocabulary size:", VOCAB_SIZE_SYMBOLS)

Repeat vocabulary size: 41507
Repeat + symbols removed vocabulary size: 42805


In [ ]:
def encode_text_with_vocab(text, vocab, max_len, pad_idx, unk_idx):
  tokens = tokenize(text)
  ids = [
      vocab.get(token, unk_idx)
      for token in tokens
  ]

  ids = ids[:max_len]

  if len(ids) < max_len:
    ids += [pad_idx] * (max_len - len(ids))

  return ids

In [ ]:
class KOTEDatasetV2(Dataset):

  def __init__(self, texts, labels, vocab, max_len, pad_idx, unk_idx):
    self.texts = texts.reset_index(drop=True)
    self.labels = labels
    self.vocab = vocab
    self.max_len = max_len
    self.pad_idx = pad_idx
    self.unk_idx = unk_idx

  def __len__(self):
    return len(self.texts)

  def __getitem__(self, idx):
    input_ids = encode_text_with_vocab(
        self.texts.iloc[idx],
        self.vocab,
        self.max_len,
        self.pad_idx,
        self.unk_idx
    )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(self.labels[idx], dtype=torch.float32)
    }

In [ ]:
train_dataset_symbols = KOTEDatasetV2(
    train_texts_symbols,
    y_train,
    vocab_symbols,
    MAX_LEN,
    PAD_IDX_SYMBOLS,
    UNK_IDX_SYMBOLS
)

val_dataset_symbols = KOTEDatasetV2(
    val_texts_symbols,
    y_train,
    vocab_symbols,
    MAX_LEN,
    PAD_IDX_SYMBOLS,
    UNK_IDX_SYMBOLS
)

test_dataset_symbols = KOTEDatasetV2(
    test_texts_symbols,
    y_train,
    vocab_symbols,
    MAX_LEN,
    PAD_IDX_SYMBOLS,
    UNK_IDX_SYMBOLS
)

In [ ]:
train_loader_symbols = DataLoader(
    train_dataset_symbols,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader_symbols = DataLoader(
    val_dataset_symbols,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader_symbols = DataLoader(
    test_dataset_symbols,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
model_symbols = TextCNN(
    vocab_size=VOCAB_SIZE_SYMBOLS,
    embed_dim=EMBED_DIM,
    num_filters=NUM_FILTERS,
    kernel_sizes=KERNEL_SIZES,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
    padding_idx=PAD_IDX_SYMBOLS
).to(device)

criterion_symbols = nn.BCEWithLogitsLoss()

optimizer_symbols = torch.optim.Adam(
    model_symbols.parameters(),
    lr=1e-3
)

In [ ]:
EPOCHS = 5

history = []

for epoch in range(1, EPOCHS + 1):
  train_loss = train_one_epoch(
      model_symbols,
      train_loader_symbols,
      criterion_symbols,
      optimizer_symbols,
      device
  )

  val_probs_symbols, val_labels_symbols = predict(
      model_symbols,
      val_loader_symbols,
      device
  )

  result = evaluate_multilabel(
      val_labels_symbols,
      val_probs_symbols,
      threshold=0.5
  )

  history.append({
      "epoch": epoch,
      "train_loss": train_loss,
      **result
  })

  print(
      f"Epoch {epoch:02d} | "
      f"Loss: {train_loss:.4f} | "
      f"Micro F1: {result['micro_f1']:.4f} | "
      f"Macro F1: {result['macro_f1']:.4f}"
  )

Epoch 01 | Loss: 0.3847 | Micro F1: 0.1824 | Macro F1: 0.0746
Epoch 02 | Loss: 0.3731 | Micro F1: 0.1790 | Macro F1: 0.0831
Epoch 03 | Loss: 0.3636 | Micro F1: 0.1809 | Macro F1: 0.0842
Epoch 04 | Loss: 0.3559 | Micro F1: 0.1770 | Macro F1: 0.0821
Epoch 05 | Loss: 0.3484 | Micro F1: 0.1897 | Macro F1: 0.0895


In [ ]:
thresholds = np.arange(0.10, 0.61, 0.05)

threshold_results_symbols = []

for threshold in thresholds:
  result = evaluate_multilabel(
      val_labels_symbols,
      val_probs_symbols,
      threshold=threshold
  )
  threshold_results_symbols.append(result)

threshold_df_symbols = pd.DataFrame(threshold_results_symbols)

threshold_df_symbols

,threshold,micro_f1,macro_f1,hamming_loss,pr_auc_micro,pr_auc_macro
0,0.10,0.340920,0.263354,0.464014,0.259299,0.180329
1,0.15,0.343432,0.239002,0.372573,0.259299,0.180329
2,0.20,0.333494,0.211385,0.314882,0.259299,0.180329
3,0.25,0.312511,0.180657,0.276886,0.259299,0.180329
4,0.30,0.289033,0.156246,0.251818,0.259299,0.180329
5,0.35,0.264826,0.137220,0.234855,0.259299,0.180329
6,0.40,0.239635,0.119839,0.222832,0.259299,0.180329
7,0.45,0.211993,0.102703,0.214141,0.259299,0.180329
8,0.50,0.189709,0.089540,0.207000,0.259299,0.180329
9,0.55,0.166529,0.077407,0.201745,0.259299,0.180329


In [ ]:
best_idx_symbols = threshold_df_symbols["micro_f1"].idxmax()
best_result_symbols = threshold_df_symbols.loc[best_idx_symbols]

print("Best threshold:", best_result_symbols["threshold"])
print("Micro F1:", best_result_symbols["micro_f1"])
print("Macro F1:", best_result_symbols["macro_f1"])
print("Hamming Loss:", best_result_symbols["hamming_loss"])
print("PR-AUC Micro:", best_result_symbols["pr_auc_micro"])
print("PR-AUC Macro:", best_result_symbols["pr_auc_macro"])

Best threshold: 0.15000000000000002
Micro F1: 0.34343159243832105
Macro F1: 0.2390021898501682
Hamming Loss: 0.3725727272727273
PR-AUC Micro: 0.2592987153932487
PR-AUC Macro: 0.18032940823720825


In [ ]:
for i in range(20):
    print("ORIGINAL       :", train_df["text"].iloc[i])
    print("REPEAT         :", train_texts.iloc[i])
    print("REPEAT_SYMBOLS :", train_texts_symbols.iloc[i])
    print("-" * 80)

ORIGINAL       : 내가 톰행크스를 좋아하긴 했나보다... 초기 영화 빼고는 다 봤네.
REPEAT         : 내가 톰행크스를 좋아하긴 했나보다.. 초기 영화 빼고는 다 봤네.
REPEAT_SYMBOLS : 내가 톰행크스를 좋아하긴 했나보다 초기 영화 빼고는 다 봤네
--------------------------------------------------------------------------------
ORIGINAL       : 정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요 ㅠ 생각보다 스트레스 심한 직업군이라는 생각입니다.
REPEAT         : 정말 상상을 초월하는 무개념 진상들 상대하다 우울증, 공항장애 걸리는 공무원 많아요 ㅠ 생각보다 스트레스 심한 직업군이라는 생각입니다.
REPEAT_SYMBOLS : 정말 상상을 초월하는 무개념 진상들 상대하다 우울증 공항장애 걸리는 공무원 많아요 ㅠ 생각보다 스트레스 심한 직업군이라는 생각입니다
--------------------------------------------------------------------------------
ORIGINAL       : 새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수함일지도 모르겠습니다. 그걸 가능케 한 사랑에 경의를 표합니다.
REPEAT         : 새로운 세상과 조우한 자의 어린아이 같은 반응, 어쩌면 회복된 것은 눈이 아닌 순수함일지도 모르겠습니다. 그걸 가능케 한 사랑에 경의를 표합니다.
REPEAT_SYMBOLS : 새로운 세상과 조우한 자의 어린아이 같은 반응 어쩌면 회복된 것은 눈이 아닌 순수함일지도 모르겠습니다 그걸 가능케 한 사랑에 경의를 표합니다
--------------------------------------------------------------------------------
ORIGINAL       : 미역은 원생생물계

In [ ]:
print("repeat vocab size:", len(vocab))
print("repeat_symbols vocab size:", len(vocab_symbols))

repeat vocab size: 41507
repeat_symbols vocab size: 42805


In [ ]:
def get_unk_ratio(texts, vocab):
    total_tokens = 0
    unk_tokens = 0

    for text in texts:
        tokens = tokenize(text)

        total_tokens += len(tokens)
        unk_tokens += sum(
            token not in vocab
            for token in tokens
        )

    return unk_tokens / total_tokens


print(
    "repeat val UNK ratio:",
    get_unk_ratio(val_texts, vocab)
)

print(
    "repeat_symbols val UNK ratio:",
    get_unk_ratio(val_texts_symbols, vocab_symbols)
)

repeat val UNK ratio: 0.43479499762545515
repeat_symbols val UNK ratio: 0.3925142467789891


In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

model_symbols_clean = TextCNN(
    vocab_size=VOCAB_SIZE_SYMBOLS,
    embed_dim=EMBED_DIM,
    num_filters=NUM_FILTERS,
    kernel_sizes=KERNEL_SIZES,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
    padding_idx=PAD_IDX_SYMBOLS
).to(device)

criterion_symbols_clean = nn.BCEWithLogitsLoss()

optimizer_symbols_clean = torch.optim.Adam(
    model_symbols_clean.parameters(),
    lr=1e-3
)

In [ ]:
EPOCHS = 5

for epoch in range(1, EPOCHS + 1):

    train_loss = train_one_epoch(
        model_symbols_clean,
        train_loader_symbols,
        criterion_symbols_clean,
        optimizer_symbols_clean,
        device
    )

    val_probs_symbols_clean, val_labels_symbols_clean = predict(
        model_symbols_clean,
        val_loader_symbols,
        device
    )

    result = evaluate_multilabel(
        val_labels_symbols_clean,
        val_probs_symbols_clean,
        threshold=0.5
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Micro F1: {result['micro_f1']:.4f} | "
        f"Macro F1: {result['macro_f1']:.4f}"
    )

Epoch 01 | Loss: 0.4453 | Micro F1: 0.1062 | Macro F1: 0.0423
Epoch 02 | Loss: 0.4122 | Micro F1: 0.1636 | Macro F1: 0.0625
Epoch 03 | Loss: 0.3939 | Micro F1: 0.1585 | Macro F1: 0.0690
Epoch 04 | Loss: 0.3800 | Micro F1: 0.1736 | Macro F1: 0.0813
Epoch 05 | Loss: 0.3688 | Micro F1: 0.1660 | Macro F1: 0.0736


In [ ]:
threshold_results_symbols_clean = []

for threshold in np.arange(0.10, 0.61, 0.05):

    result = evaluate_multilabel(
        val_labels_symbols_clean,
        val_probs_symbols_clean,
        threshold=threshold
    )

    threshold_results_symbols_clean.append(result)

threshold_df_symbols_clean = pd.DataFrame(
    threshold_results_symbols_clean
)

threshold_df_symbols_clean

,threshold,micro_f1,macro_f1,hamming_loss,pr_auc_micro,pr_auc_macro
0,0.10,0.347528,0.262966,0.459536,0.265222,0.181406
1,0.15,0.347572,0.233050,0.365864,0.265222,0.181406
2,0.20,0.335706,0.203813,0.305886,0.265222,0.181406
3,0.25,0.311869,0.172754,0.266000,0.265222,0.181406
4,0.30,0.285195,0.144393,0.239745,0.265222,0.181406
5,0.35,0.250053,0.120877,0.224282,0.265222,0.181406
6,0.40,0.223163,0.104120,0.212627,0.265222,0.181406
7,0.45,0.191931,0.087172,0.204655,0.265222,0.181406
8,0.50,0.165991,0.073641,0.199195,0.265222,0.181406
9,0.55,0.142025,0.062428,0.194850,0.265222,0.181406


In [ ]:
best_idx = threshold_df_symbols_clean["micro_f1"].idxmax()
best = threshold_df_symbols_clean.loc[best_idx]

print("Best threshold:", best["threshold"])
print("Micro F1:", best["micro_f1"])
print("Macro F1:", best["macro_f1"])
print("Hamming Loss:", best["hamming_loss"])
print("PR-AUC Micro:", best["pr_auc_micro"])
print("PR-AUC Macro:", best["pr_auc_macro"])

Best threshold: 0.15000000000000002
Micro F1: 0.34757234335738024
Macro F1: 0.23305009051060466
Hamming Loss: 0.3658636363636364
PR-AUC Micro: 0.26522204988575243
PR-AUC Macro: 0.18140626173294694
